# Week 13 · Optimizer sanity and score-training stability

**Mode: experiment · two sessions · four implementation exercises**

**Question:** How do optimizer state and learning rate change a training
trajectory, and what can a controlled intervention tell us about stability?

Keep this notebook as the baseline. Work in
**`w13_optimizers_sanity_solved.ipynb`**, which starts as an exact copy,
with the same exercises. It is your working copy rather than a filled answer key.

| Session | Route | Stop condition |
| --- | --- | --- |
| **6 · today · 75–90 min** | Three update-rule TODOs → hand checks → trajectories/loss/gradient norms → rate comparisons | Explain a failure and a successful run from their update rules |
| **7 · next session · 75–90 min** | Independent score-toy setup → clipping TODO → paired experiment → interpretation | Distinguish bounded gradients from improved held-out score fit |

Session 6 has **three small update-rule TODOs**. The objective, experiment
runner, plots, and assertions are supplied. Session 7 has **one clipping TODO**.
The assertions become runnable after you fill the corresponding exercise.

Run cells sequentially as you work. **Stop at “Session 6 stop”.** Session 7 is
disabled by `RUN_SESSION_7 = False`; set it to `True` when you return. Its setup
is independent, so you can restart the kernel and begin at that section tomorrow.

Reading companions: [optimization notes](../notes/w13_optimization.md),
[hand calculations](../proofs/w13_optimizer_updates.md), and
[Week 13 curriculum](../CURRICULUM.md#week-13--optimization-basics-just-enough).
The quadratic and experiments are our independent practice work.

## Session 6 · 1. Fix the objective and starting point

Use the same objective as the hand calculations:

$$
L(\theta)=\frac12(\theta_1^2+10\theta_2^2),\qquad
\nabla L(\theta)=(\theta_1,10\theta_2)^\top,\qquad
\theta^{(0)}=(2,1)^\top.
$$

These gradients are exact and deterministic. The seed below only makes our
numerical gradient-check point reproducible; it does not make the optimizer stochastic.
NumPy arrays represent the two coordinates as shape `(2,)`.

In [ ]:
from pathlib import Path
import os
import tempfile

os.environ.setdefault('MPLCONFIGDIR', str(Path(tempfile.gettempdir()) / 'w13-matplotlib'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'CURRICULUM.md').exists())
S6 = {
    'seed': 13,
    'theta0': np.array([2.0, 1.0]),
    'hand_lr': 0.05,
    'steps': 120,
    'momentum': 0.9,
    'beta1': 0.9,
    'beta2': 0.999,
    'eps': 1e-8,
}
S6_CURVATURE = np.array([1.0, 10.0])
s6_rng = np.random.default_rng(S6['seed'])

plt.rcParams.update({'figure.dpi': 110, 'font.size': 10, 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.facecolor': 'white'})

def quadratic_loss(theta: np.ndarray) -> np.ndarray:
    """Evaluate the loss along the last coordinate axis."""
    values = np.asarray(theta, dtype=float)
    return 0.5 * np.sum(S6_CURVATURE * values**2, axis=-1)

def quadratic_gradient(theta: np.ndarray) -> np.ndarray:
    """Evaluate the exact gradient along the last coordinate axis."""
    return S6_CURVATURE * np.asarray(theta, dtype=float)

assert quadratic_loss(S6['theta0']) == 7.0
np.testing.assert_array_equal(quadratic_gradient(S6['theta0']), [2.0, 10.0])
assert quadratic_loss(np.zeros(2)) == 0.0

# Check the supplied gradient independently with central finite differences.
check_point = s6_rng.normal(size=2)
h = 1e-6
numeric_gradient = np.array([
    (quadratic_loss(check_point + h * e) - quadratic_loss(check_point - h * e)) / (2*h)
    for e in np.eye(2)
])
np.testing.assert_allclose(numeric_gradient, quadratic_gradient(check_point), rtol=1e-6, atol=1e-7)
print('Objective and gradient checks passed.')

## Session 6 · 2. Implement and verify the three update rules

### TODO 1 of 4 · Gradient descent

Implement one update. The runner will supply the gradient at the current state:

$$
\theta^{(k+1)}=\theta^{(k)}-\eta g^{(k)}.
$$

Return a new parameter vector. Keep the input arrays unchanged so later
comparisons can reuse the same starting state.

In [ ]:
def gd_step(theta: np.ndarray, g: np.ndarray, *, lr: float) -> np.ndarray:
    """Return one gradient-descent update."""
    # TODO 1: implement the parameter update in NumPy.
    raise NotImplementedError('TODO 1: implement gd_step in your working copy.')

In [ ]:
theta_gd_0 = S6['theta0'].copy()
theta_gd_1 = gd_step(theta_gd_0, quadratic_gradient(theta_gd_0), lr=S6['hand_lr'])
theta_gd_2 = gd_step(theta_gd_1, quadratic_gradient(theta_gd_1), lr=S6['hand_lr'])
np.testing.assert_array_equal(theta_gd_0, S6['theta0'])
np.testing.assert_allclose(theta_gd_1, [1.9, 0.5], atol=1e-12)
np.testing.assert_allclose(theta_gd_2, [1.805, 0.25], atol=1e-12)
print('GD reproduces both handwritten updates.')

### TODO 2 of 4 · Momentum in the displacement convention

Use the MML convention we chose, with $\Delta^{(0)}=0$:

$$
\Delta^{(k+1)}=\alpha\Delta^{(k)}-\eta g^{(k)},\qquad
\theta^{(k+1)}=\theta^{(k)}+\Delta^{(k+1)}.
$$

`delta` stores a signed parameter displacement. Return both the new parameter
vector and new displacement; the old state must remain unchanged.

In [ ]:
def momentum_step(
    theta: np.ndarray, g: np.ndarray, delta: np.ndarray,
    *, lr: float, alpha: float,
) -> tuple[np.ndarray, np.ndarray]:
    """Return updated parameters and displacement state."""
    # TODO 2: compute the next displacement, then the next parameters.
    raise NotImplementedError('TODO 2: implement momentum_step in your working copy.')

In [ ]:
theta_mom_0 = S6['theta0'].copy()
delta_0 = np.zeros(2)
theta_mom_1, delta_1 = momentum_step(
    theta_mom_0, quadratic_gradient(theta_mom_0), delta_0,
    lr=S6['hand_lr'], alpha=S6['momentum'],
)
theta_mom_2, delta_2 = momentum_step(
    theta_mom_1, quadratic_gradient(theta_mom_1), delta_1,
    lr=S6['hand_lr'], alpha=S6['momentum'],
)
np.testing.assert_array_equal(theta_mom_0, S6['theta0'])
np.testing.assert_array_equal(delta_0, [0.0, 0.0])
np.testing.assert_allclose(delta_1, [-0.1, -0.5], atol=1e-12)
np.testing.assert_allclose(theta_mom_1, [1.9, 0.5], atol=1e-12)
np.testing.assert_allclose(delta_2, [-0.185, -0.7], atol=1e-12)
np.testing.assert_allclose(theta_mom_2, [1.715, -0.2], atol=1e-12)
print('Momentum reproduces both handwritten updates and states.')

### TODO 3 of 4 · Adam, with the unified indexing

`g` is evaluated at $\theta^{(k)}$. The argument `step` is **$k+1$**, the
number of gradients incorporated into the new moment states:

$$
\begin{aligned}
m^{(k+1)}&=\beta_1m^{(k)}+(1-\beta_1)g^{(k)},\\
v^{(k+1)}&=\beta_2v^{(k)}+(1-\beta_2)(g^{(k)}\odot g^{(k)}),\\
\widehat m^{(k+1)}&=m^{(k+1)}/(1-\beta_1^{k+1}),\\
\widehat v^{(k+1)}&=v^{(k+1)}/(1-\beta_2^{k+1}),\\
\theta^{(k+1)}&=\theta^{(k)}-\eta\widehat m^{(k+1)}/
\bigl(\sqrt{\widehat v^{(k+1)}}+\epsilon\bigr).
\end{aligned}
$$

Return the new parameters and **uncorrected** new `m` and `v` states.
All operations on coordinates are elementwise. Put $\epsilon$ outside the
square root. Reference: [Adam, Algorithm 1](https://arxiv.org/pdf/1412.6980).

In [ ]:
def adam_step(
    theta: np.ndarray, g: np.ndarray, m: np.ndarray, v: np.ndarray,
    *, step: int, lr: float, beta1: float, beta2: float, eps: float,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return updated parameters and uncorrected moment states."""
    assert step >= 1, 'Bias correction counts observations starting at 1.'
    # TODO 3: update m/v, compute their corrected versions, update theta.
    raise NotImplementedError('TODO 3: implement adam_step in your working copy.')

In [ ]:
theta_adam_0 = S6['theta0'].copy()
m0, v0 = np.zeros(2), np.zeros(2)
adam_kwargs = {key: S6[key] for key in ('beta1', 'beta2', 'eps')}
theta_adam_1, m1, v1 = adam_step(
    theta_adam_0, quadratic_gradient(theta_adam_0), m0, v0,
    step=1, lr=S6['hand_lr'], **adam_kwargs,
)
theta_adam_2, m2, v2 = adam_step(
    theta_adam_1, quadratic_gradient(theta_adam_1), m1, v1,
    step=2, lr=S6['hand_lr'], **adam_kwargs,
)
np.testing.assert_array_equal(theta_adam_0, S6['theta0'])
np.testing.assert_array_equal(m0, [0.0, 0.0])
np.testing.assert_array_equal(v0, [0.0, 0.0])
np.testing.assert_allclose(m1, [0.2, 1.0], atol=1e-12)
np.testing.assert_allclose(v1, [0.004, 0.1], atol=1e-12)
np.testing.assert_allclose(theta_adam_1, [1.950000000250, 0.950000000050], rtol=0, atol=5e-12)
np.testing.assert_allclose(theta_adam_2, [1.900036997917, 0.900083242906], rtol=0, atol=5e-12)
np.testing.assert_allclose(m2, [0.375, 1.85], rtol=0, atol=1e-9)
np.testing.assert_allclose(v2, [0.0077985, 0.19015], rtol=0, atol=1e-9)

# A deliberately large epsilon isolates the parentheses, which 1e-8 barely exposes.
theta_eps, _, _ = adam_step(
    S6['theta0'].copy(), np.array([2.0, 10.0]), np.zeros(2), np.zeros(2),
    step=1, lr=0.05, beta1=0.9, beta2=0.999, eps=0.5,
)
np.testing.assert_allclose(theta_eps, [1.96, 0.9523809523809523], rtol=0, atol=1e-12)
print('Adam reproduces both worksheet updates, raw states, and epsilon placement.')

## Session 6 · 3. Run each optimizer from a fresh state

The supplied runner recomputes the gradient at every new parameter vector and
resets optimizer memory for every trial. Row $0$ is the initial state; row $k$
records the loss and full-gradient norm at $\theta^{(k)}$.

Begin with the shared $\eta=0.05$ to connect code to your calculations. This
is a controlled illustration, not a claim that equal rates make a fair
performance comparison. The later grids compare rates separately.
The summary's `finite` status only means the values remain representable;
it does not establish convergence.

In [ ]:
def run_optimizer(method: str, lr: float, steps: int | None = None) -> dict:
    """Run an optimizer with fresh memory and record each parameter state."""
    assert method in {'gd', 'momentum', 'adam'}
    assert lr > 0
    steps = S6['steps'] if steps is None else steps
    theta = S6['theta0'].copy()
    delta, m, v = np.zeros(2), np.zeros(2), np.zeros(2)
    positions = [theta.copy()]
    status = 'finite'
    for k in range(steps):
        g = quadratic_gradient(theta)
        with np.errstate(over='ignore', invalid='ignore'):
            if method == 'gd':
                theta = gd_step(theta, g, lr=lr)
            elif method == 'momentum':
                theta, delta = momentum_step(theta, g, delta, lr=lr, alpha=S6['momentum'])
            else:
                theta, m, v = adam_step(theta, g, m, v, step=k+1, lr=lr,
                                       beta1=S6['beta1'], beta2=S6['beta2'], eps=S6['eps'])
        theta = np.asarray(theta, dtype=float)
        assert theta.shape == (2,)
        positions.append(theta.copy())
        if not np.isfinite(theta).all():
            status = 'non-finite parameters; stopped'
            break
    positions = np.array(positions)
    with np.errstate(over='ignore', invalid='ignore'):
        losses = quadratic_loss(positions)
        grad_norms = np.linalg.norm(quadratic_gradient(positions), axis=1)
    return {'method': method, 'lr': lr, 'theta': positions,
            'iteration': np.arange(len(positions)), 'loss': losses,
            'grad_norm': grad_norms,
            'step_norm': np.linalg.norm(np.diff(positions, axis=0), axis=1),
            'status': status}

METHOD_NAMES = {'gd': 'GD', 'momentum': 'Momentum', 'adam': 'Adam'}
METHOD_COLORS = {'gd': '#008b98', 'momentum': '#7954b7', 'adam': '#c9553c'}

def run_summary(runs: list[dict]) -> pd.DataFrame:
    """Summarize a fixed-budget set of trials."""
    return pd.DataFrame([{
        'optimizer': METHOD_NAMES[r['method']], 'lr': r['lr'],
        'updates': len(r['iteration'])-1, 'final_loss': r['loss'][-1],
        'min_loss': np.nanmin(r['loss']), 'final_grad_norm': r['grad_norm'][-1],
        'loss_increases': int(np.sum(np.diff(r['loss']) > 1e-12)),
        'status': r['status'],
    } for r in runs])

baseline_runs = [run_optimizer(name, S6['hand_lr']) for name in ('gd', 'momentum', 'adam')]
assert all(len(r['iteration']) == S6['steps']+1 for r in baseline_runs)
assert all(np.isfinite(r['loss']).all() for r in baseline_runs)
assert all(r['loss'][0] == 7.0 for r in baseline_runs)
display(run_summary(baseline_runs))

In [ ]:
def plot_quadratic_runs(runs: list[dict], title: str, *, first_steps: int | None = None) -> None:
    """Show contour trajectories, losses, and full-gradient norms."""
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), layout='constrained')
    x_limits = (-2.5, 2.5) if first_steps is None else (-4.2, 4.2)
    y_limits = (-1.2, 1.2) if first_steps is None else (-3.6, 3.6)
    x = np.linspace(*x_limits, 240)
    y = np.linspace(*y_limits, 240)
    xx, yy = np.meshgrid(x, y)
    zz = quadratic_loss(np.stack((xx, yy), axis=-1))
    axes[0].contour(xx, yy, zz, levels=[0.05, 0.2, 0.5, 1, 2, 4, 7, 10],
                    colors='#bac7ce', linewidths=0.8)
    axes[0].axhline(0, color='#acb7bc', lw=0.7)
    axes[0].axvline(0, color='#acb7bc', lw=0.7)
    axes[0].scatter(0, 0, marker='*', color='#d3a446', edgecolor='#334956', s=120, zorder=5)
    axes[0].scatter(*S6['theta0'], color='#334956', s=30, zorder=5)
    for index, run in enumerate(runs):
        label = f"{METHOD_NAMES[run['method']]} · lr={run['lr']:g}"
        color = (METHOD_COLORS[run['method']] if len({r['method'] for r in runs}) > 1
                 else ['#008b98', '#7954b7', '#c9553c'][index])
        points = run['theta'] if first_steps is None else run['theta'][:first_steps+1]
        axes[0].plot(points[:, 0], points[:, 1], color=color, lw=1.7, label=label)
        axes[0].scatter(points[1:4, 0], points[1:4, 1], s=18, color=color, zorder=4)
        axes[1].semilogy(run['iteration'], np.maximum(run['loss'], 1e-30), color=color, label=label)
        axes[2].semilogy(run['iteration'], np.maximum(run['grad_norm'], 1e-30), color=color, label=label)
    axes[0].set(xlim=x_limits, ylim=y_limits, xlabel=r'$\theta_1$', ylabel=r'$\theta_2$')
    axes[0].set_aspect('equal', adjustable='box')
    axes[0].set_title('Parameter trajectory' if first_steps is None else f'First {first_steps} updates')
    axes[1].set(xlabel='Update number', ylabel='Loss', title='Loss (log scale)')
    axes[2].set(xlabel='Update number', ylabel=r'$\|\nabla L\|_2$', title='Gradient norm (log scale)')
    for ax in axes:
        ax.legend(fontsize=8, loc='best')
        ax.grid(alpha=0.15)
    fig.suptitle(title)
    plt.show()

plot_quadratic_runs(baseline_runs, 'Shared hand-calculation rate: 0.05')

## Session 6 · 4. Predict and inspect controlled divergence

For GD, the coordinate multipliers are $1-\eta$ and $1-10\eta$.
Before running the next cell, predict the behavior for $\eta=0.05$, $0.15$,
and $0.25$: do signs change, and do coordinate magnitudes shrink or grow?

The contour window shows the first three updates for all three rates.
The divergent trajectory subsequently leaves that window; the loss and
gradient-norm panels retain its whole 25-update history.
No gradient clipping is applied to this quadratic experiment.

In [ ]:
gd_failure_runs = [run_optimizer('gd', lr, steps=25) for lr in (0.05, 0.15, 0.25)]
for run in gd_failure_runs:
    k = run['iteration'][:, None]
    expected = S6['theta0'] * (1-run['lr']*S6_CURVATURE)**k
    np.testing.assert_allclose(run['theta'], expected, rtol=1e-10, atol=1e-12)
np.testing.assert_allclose(gd_failure_runs[-1]['loss'][1], 12.375)
assert gd_failure_runs[-1]['loss'][-1] > gd_failure_runs[-1]['loss'][0]
assert gd_failure_runs[-1]['grad_norm'][-1] > gd_failure_runs[-1]['grad_norm'][0]
assert gd_failure_runs[0]['loss'][-1] < 7 and gd_failure_runs[1]['loss'][-1] < 7
plot_quadratic_runs(gd_failure_runs, 'Controlled GD failure · contour window is fixed', first_steps=3)
display(run_summary(gd_failure_runs))

## Session 6 · 5. Give each optimizer a small learning-rate comparison

Each optimizer gets three trials and the same 120-update budget. GD's grid
includes our known failure. Momentum and Adam use $0.01$, $0.05$, and $0.15$
to inspect a smaller rate, the hand-calculation rate, and a larger rate.
Keep momentum and Adam's other coefficients fixed.

This is a small diagnostic comparison, not exhaustive tuning or a wall-clock
benchmark. Compare sensitivity **within** each optimizer before comparing
their best observed losses. A gradient norm measures the objective's slope;
it is not generally the norm of the parameter update.

In [ ]:
S6_LR_GRID = {'gd': (0.05, 0.15, 0.25),
              'momentum': (0.01, 0.05, 0.15),
              'adam': (0.01, 0.05, 0.15)}
sweep_runs = {name: [run_optimizer(name, lr) for lr in rates]
              for name, rates in S6_LR_GRID.items()}
assert all(len(r['iteration']) == S6['steps']+1 for runs in sweep_runs.values() for r in runs)
assert all(np.isfinite(r['loss']).all() for runs in sweep_runs.values() for r in runs)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), layout='constrained')
for ax, (name, runs) in zip(axes, sweep_runs.items()):
    for color, run in zip(['#008b98', '#7954b7', '#c9553c'], runs):
        ax.semilogy(run['iteration'], np.maximum(run['loss'], 1e-30), color=color,
                    label=f"lr={run['lr']:g}")
    ax.set(title=METHOD_NAMES[name], xlabel='Update number', ylabel='Loss (log scale)')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.2)
fig.suptitle('Three rates per optimizer · fixed 120-update budget')
plt.show()
display(run_summary([r for runs in sweep_runs.values() for r in runs]))

## Session 6 stop · Explain the behavior before moving on

Save your work in the `_solved` notebook. Write short answers here and come
back to our chat to review them. **Your study session ends here today.**

1. Did each optimizer reproduce the first two hand-calculated updates?
   If something failed, identify the index, state, or operation that caused it.
   **My check:** [your answer]
2. Explain GD at $0.05$, $0.15$, and $0.25$ using the coordinate multipliers.
   Why does the divergent run also develop large gradient norms?
   **My explanation:** [your answer]
3. Choose one momentum or Adam trajectory and explain it from the update rule.
   What does its learning-rate grid show, and why is equal numerical learning
   rate insufficient to establish an overall winner?
   **My interpretation:** [your answer]

**Tomorrow:** restart the kernel, scroll to Session 7, and change
`RUN_SESSION_7` to `True` in its setup cell. Session 7 does not require the
Session 6 functions or their outputs.

## Session 7 · 1. Prepare a controlled score-training experiment

**Question:** With the optimizer and training conditions matched, does global
gradient-norm clipping improve the learned score, or only bound the gradients?

We reuse Week 8's direct-score MLP, Gaussian-blob data construction, fixed
noise level, SGD learning rate $0.1$, and 20-update budget. This is a new
paired experiment built from that code. It uses CPU for repeatability.
The original training and evaluation scripts remain available as references:
[training loop](../mini_projects/checkpoint_02_toy_score_matching/train.py)
and [analytical score](../mini_projects/checkpoint_02_toy_score_matching/evaluate.py).

One condition changes: clipping is off or has a **fixed global norm threshold
of 0.5**. During notebook preparation, the untrained model's first gradient
norm was about 0.83; we chose 0.5 to make the mechanical effect visible.
The threshold was selected from that gradient scale, rather than score-fit
performance. Set your fit hypothesis before running the paired experiment.
If clipping has little effect on fit, or worsens it, report that observation.

In [ ]:
RUN_SESSION_7 = False  # Change to True when you return for Session 7.
if not RUN_SESSION_7:
    raise RuntimeError('Session 6 stop: save your answers and return to the chat. Enable Session 7 when ready.')

# Independent setup: this cell can run in a fresh kernel.
from pathlib import Path
from copy import deepcopy
from types import SimpleNamespace
import os
import sys
import tempfile

os.environ.setdefault('MPLCONFIGDIR', str(Path(tempfile.gettempdir()) / 'w13-matplotlib'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from matplotlib.ticker import MaxNLocator
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'CURRICULUM.md').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from mini_projects.checkpoint_02_toy_score_matching.model import BlobsMLP
from mini_projects.checkpoint_02_toy_score_matching.train import (
    create_toy_dataset, CENTERS, CLUSTER_STD, NUM_SAMPLES, TRAIN_PERCENTAGE,
    FORWARD_SIGMA, TRAIN_CONFIG, MAX_ITERS, RANDOM_SEED,
)
from mini_projects.checkpoint_02_toy_score_matching.evaluate import gaussian_mixture_score

torch.set_num_threads(2)
S7 = {'data_seed': RANDOM_SEED, 'split_seed': RANDOM_SEED, 'init_seed': RANDOM_SEED,
      'noise_seed': 43, 'heldout_noise_seed': 44, 'num_samples': NUM_SAMPLES,
      'train_fraction': TRAIN_PERCENTAGE, 'sigma': FORWARD_SIGMA,
      'lr': TRAIN_CONFIG['learning_rate'], 'steps': MAX_ITERS, 'max_norm': 0.5}
S7_DEVICE = torch.device('cpu')
assert S7['max_norm'] > 0 and S7['steps'] > 0
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10, 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.facecolor': 'white'})
display(pd.Series(S7, name='Matched experiment configuration').to_frame())

### Freeze the conditions shared by both runs

The clean training points are reused at every update, as in Week 8.
We precompute **one sequence of fresh corruption draws** and give that exact
sequence to both runs. Held-out clean points and their noisy versions are fixed.
Each run starts from a deep copy of one untrained model.

Training uses the conditional Gaussian score target $-\epsilon/\sigma$.
Evaluation compares predictions with the **marginal noisy-mixture score**,
whose component standard deviation is

$$
\sigma_{\mathrm{mixture,noisy}}
=\sqrt{\sigma_{\mathrm{clean}}^2+\sigma_{\mathrm{corruption}}^2}.
$$

These two targets play different roles. Both MSE values average over points
and the two data coordinates.

In [ ]:
s7_rng = np.random.default_rng(S7['split_seed'])
s7_clean, _ = create_toy_dataset(
    num_samples=S7['num_samples'], centers=CENTERS,
    cluster_std=CLUSTER_STD, random_seed=S7['data_seed'],
)
s7_indices = s7_rng.permutation(len(s7_clean))
s7_train_size = int(S7['train_fraction'] * len(s7_clean))
s7_train_indices = s7_indices[:s7_train_size]
s7_val_indices = s7_indices[s7_train_size:]
assert set(s7_train_indices).isdisjoint(set(s7_val_indices))
s7_train_x0 = torch.tensor(s7_clean[s7_train_indices], dtype=torch.float32, device=S7_DEVICE)
s7_val_x0 = torch.tensor(s7_clean[s7_val_indices], dtype=torch.float32, device=S7_DEVICE)

s7_noise_generator = torch.Generator(device='cpu').manual_seed(S7['noise_seed'])
s7_train_noise = torch.randn(
    (S7['steps'], *s7_train_x0.shape), generator=s7_noise_generator, device=S7_DEVICE,
)
s7_val_generator = torch.Generator(device='cpu').manual_seed(S7['heldout_noise_seed'])
s7_val_noise = torch.randn(s7_val_x0.shape, generator=s7_val_generator, device=S7_DEVICE)
s7_val_xt = s7_val_x0 + S7['sigma'] * s7_val_noise
noisy_component_std = np.sqrt(CLUSTER_STD**2 + S7['sigma']**2)
s7_true_val_score = torch.tensor(
    gaussian_mixture_score(s7_val_xt.numpy(), np.asarray(CENTERS), noisy_component_std),
    dtype=torch.float32, device=S7_DEVICE,
)

torch.manual_seed(S7['init_seed'])
s7_model_config = SimpleNamespace(**{**TRAIN_CONFIG, 'objective': 'score'})
s7_initial_model = BlobsMLP(s7_model_config).to(S7_DEVICE)
s7_initial_state = {name: tensor.detach().clone() for name, tensor in s7_initial_model.state_dict().items()}
assert s7_train_noise.shape == (S7['steps'], s7_train_size, 2)
assert torch.isfinite(s7_true_val_score).all()
np.testing.assert_allclose(noisy_component_std**2, CLUSTER_STD**2 + S7['sigma']**2)
print(f'Frozen conditions: {s7_train_size} train points, {len(s7_val_x0)} held-out points, {S7["steps"]} noise draws.')

### Predict before running the comparison

Write one short hypothesis. Distinguish the expected mechanical effect on
gradient norms from your prediction about held-out score MSE.

**My hypothesis:** [your prediction, with a reason]

Keep the chosen threshold, learning rate, noise draws, and update budget
fixed for this comparison. A changed threshold would be a separate experiment.

## Session 7 · 2. Apply global gradient-norm clipping

### TODO 4 of 4 · Add the clipping operation

Fill one operation in the helper below, using
[PyTorch `clip_grad_norm_`](https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.clip_grad_norm_.html).
Clip **all model parameters together**, using the L2 norm and `max_norm`.
The operation modifies gradients in place. The supplied runner calls it
**after `loss.backward()` and before `optimizer.step()`**.

The diagnostic helper independently measures the concatenated-gradient norm
before and after this operation. When `max_norm` is `None`, the baseline
leaves gradients unchanged.

In [ ]:
def apply_global_clipping(model: nn.Module, max_norm: float | None) -> None:
    """Clip all parameter gradients together when a threshold is supplied."""
    if max_norm is None:
        return
    # TODO 4: call PyTorch's global L2 gradient-norm clipping operation.
    raise NotImplementedError('TODO 4: add clipping in your working copy.')

def global_gradient_norm(model: nn.Module) -> float:
    """Measure the L2 norm of all parameter gradients together."""
    squared_norm = sum(float(p.grad.detach().square().sum())
                       for p in model.parameters() if p.grad is not None)
    return float(np.sqrt(squared_norm))

# Isolate clipping behavior with a known gradient norm of 5.
probe = nn.Linear(2, 1, bias=False)
probe.weight.grad = torch.tensor([[3.0, 4.0]])
apply_global_clipping(probe, None)
np.testing.assert_allclose(global_gradient_norm(probe), 5.0)
apply_global_clipping(probe, 1.0)
assert global_gradient_norm(probe) <= 1.0 + 1e-5
print('Clipping bounds the global norm and leaves the off condition unchanged.')

## Session 7 · 3. Compare clipping off and on

For each update, record the sampled training loss **before** the parameter
update, gradient norms before/after clipping, and held-out score MSE **after**
the update. The initial held-out MSE is recorded separately.

The objective and optimizer remain those of the Week 8 direct-score model.
The two paths share data, corruption draws, initial parameters, SGD rate,
and budget. Clipping may change the learned parameters, so later gradients
need not match; the first gradients should match before clipping.

In [ ]:
def heldout_score_mse(model: nn.Module) -> float:
    """Compare the learned score to the known noisy-mixture score."""
    model.eval()
    with torch.no_grad():
        return float(nn.functional.mse_loss(model(s7_val_xt), s7_true_val_score))

def run_score_condition(label: str, max_norm: float | None) -> dict:
    """Train one fresh model on the shared corruption schedule."""
    model = deepcopy(s7_initial_model)
    for name, value in model.state_dict().items():
        torch.testing.assert_close(value, s7_initial_state[name], rtol=0, atol=0)
    optimizer = torch.optim.SGD(model.parameters(), lr=S7['lr'])
    initial_mse = heldout_score_mse(model)
    records = []
    for k, noise in enumerate(s7_train_noise):
        model.train()
        noisy_points = s7_train_x0 + S7['sigma'] * noise
        target = -noise / S7['sigma']
        loss = nn.functional.mse_loss(model(noisy_points), target)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        before = global_gradient_norm(model)
        assert np.isfinite(before) and torch.isfinite(loss)
        apply_global_clipping(model, max_norm)  # after backward, before step
        after = global_gradient_norm(model)
        optimizer.step()
        records.append({'update': k+1, 'train_loss': float(loss.detach()),
                        'grad_before': before, 'grad_after': after,
                        'clip_active': bool(max_norm is not None and before > max_norm),
                        'heldout_score_mse': heldout_score_mse(model)})
    return {'label': label, 'max_norm': max_norm, 'initial_mse': initial_mse,
            'history': pd.DataFrame(records), 'model': model}

s7_off = run_score_condition('Clipping off', None)
s7_on = run_score_condition('Clipping on', S7['max_norm'])
assert len(s7_off['history']) == len(s7_on['history']) == S7['steps']
np.testing.assert_allclose(s7_off['initial_mse'], s7_on['initial_mse'], rtol=0, atol=0)
np.testing.assert_allclose(s7_off['history'].iloc[0]['grad_before'],
                           s7_on['history'].iloc[0]['grad_before'], rtol=0, atol=1e-12)
np.testing.assert_allclose(s7_off['history']['grad_before'], s7_off['history']['grad_after'])
assert (s7_on['history']['grad_after'] <= S7['max_norm'] + 1e-5).all()
assert np.isfinite(s7_off['history'].select_dtypes(include='number').to_numpy()).all()
assert np.isfinite(s7_on['history'].select_dtypes(include='number').to_numpy()).all()
print('Matched initial states, first gradients, and clipping bounds verified.')

In [ ]:
s7_summary = pd.DataFrame([{
    'condition': result['label'], 'updates': len(result['history']),
    'clip_count': int(result['history']['clip_active'].sum()),
    'clip_fraction': float(result['history']['clip_active'].mean()),
    'initial_score_mse': result['initial_mse'],
    'final_score_mse': result['history']['heldout_score_mse'].iloc[-1],
    'final_train_loss': result['history']['train_loss'].iloc[-1],
    'max_raw_grad': result['history']['grad_before'].max(),
    'max_applied_grad': result['history']['grad_after'].max(),
} for result in (s7_off, s7_on)])
display(s7_summary)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), layout='constrained')
for result, color in ((s7_off, '#008b98'), (s7_on, '#7954b7')):
    frame = result['history']
    axes[0].plot(frame['update'], frame['train_loss'], label=result['label'], color=color)
    axes[1].plot(frame['update'], frame['grad_before'], '--', color=color,
                 label=result['label'] + ' · before')
    axes[1].plot(frame['update'], frame['grad_after'], color=color,
                 marker='o', markersize=3, markevery=2,
                 label=result['label'] + ' · after')
    axes[2].plot(np.r_[0, frame['update']],
                 np.r_[result['initial_mse'], frame['heldout_score_mse']],
                 label=result['label'], color=color)
axes[1].axhline(S7['max_norm'], color='#c9553c', lw=1, ls=':', zorder=1,
               label='Clipping threshold')
axes[0].set(title='Sampled training loss · before update', ylabel='MSE')
axes[1].set(title='Global gradient norms', ylabel='L2 norm')
axes[2].set(title='Held-out marginal score error', ylabel='Score MSE')
for ax in axes:
    ax.set_xlabel('Update number')
    ax.xaxis.set_major_locator(MaxNLocator(nbins=6, integer=True))
    ax.legend(fontsize=7)
    ax.grid(alpha=0.2)
fig.suptitle('One controlled change: clipping off versus on')
plt.show()

## Session 7 stop · Separate bounded gradients from a better score

Write your observations before drawing a conclusion. An individual seed and
a short update budget describe this experiment; they do not establish a
general ranking of clipping versus no clipping.

1. **Intervention:** How often did clipping activate? Did the applied global
   norm stay below the threshold? [your observations]
2. **Fit:** Which run had lower final held-out score MSE? How does that relate
   to the sampled training-loss curve? [your interpretation]
3. **Conclusion:** Did clipping help, hurt, or have little effect on the learned
   score under these matched conditions? [your conclusion]
4. **Limitation:** Name one limitation and one future controlled comparison.
   Keep it separate from today's result. [your limitation]

Save your work and return to the chat. We will use your hypothesis,
configuration, summary table, and observations to write
`notes/w13_training_stability_for_diffusion.md` during consolidation.